In [19]:
# !pip install deepxde

In [20]:
import deepxde as dde
import numpy as np
import matplotlib.pyplot as plt

dde.config.set_default_float("float64")

Set the default float type to float64


In [21]:
D = 0.4

def diffusion_pde(x, u, v):       # v is required, not used here
    du_dt  = dde.grad.jacobian(u, x, i=0, j=1)
    du_dx2 = dde.grad.hessian(u, x, i=0, j=0)
    return du_dt - D * du_dx2

In [22]:
space_domain = dde.geometry.Interval(0, 5)
time_domain  = dde.geometry.TimeDomain(0, 5)
geomtime     = dde.geometry.GeometryXTime(space_domain, time_domain)

In [32]:
m          = 100
sensor_pts = np.linspace(0, 5, m)
func_space = dde.data.GRF((5 - 0), length_scale=0.2, N=1000, interp="cubic")

In [33]:
bc = dde.icbc.DirichletBC(
    geomtime,
    lambda x, v, beg: np.zeros((len(x), 1)),
    lambda _, on_boundary: on_boundary
)
ic = dde.icbc.IC(
    geomtime,
    lambda x, v, beg: v,
    lambda _, on_initial: on_initial
)


In [34]:
time_data = dde.data.TimePDE(
    geomtime,
    diffusion_pde,
    [bc, ic],
    num_domain=4000,
    num_boundary=200,
    num_initial=200,
)

data = dde.data.PDEOperatorCartesianProd(
    time_data,
    func_space,
    sensor_pts,
    1000,                        # num_function → 1000 ICs
    function_variables=[0],      # IC depends only on x (index 0), not t (index 1)
    num_test=100,                 # optional test functions
)

In [35]:
net = dde.nn.DeepONetCartesianProd(
    layer_sizes_branch = [m, 64, 64, 64],
    layer_sizes_trunk  = [2, 64, 64, 64],
    activation         = "tanh",
    kernel_initializer = "Glorot normal",
)

In [ ]:
model = dde.Model(data, net)
model.compile("adam", lr=1e-3)
model.train(iterations=20000, display_every=1000)


Compiling model...
'compile' took 0.011441 s

Training model...



In [ ]:
ic_values = np.sin(np.pi * sensor_pts)[np.newaxis, :]  # (1, 100)
x_vals    = np.linspace(0, 5, 200)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, t_fixed in zip(axes, [0.1, 1.0, 3.0]):
    xt      = np.column_stack([x_vals, np.full_like(x_vals, t_fixed)])
    u_pred  = model.predict((ic_values, xt)).flatten()
    u_exact = np.exp(-D*np.pi**2*t_fixed) * np.sin(np.pi*x_vals)
    ax.plot(x_vals, u_exact, lw=2,         label="Exact")
    ax.plot(x_vals, u_pred,  lw=2, ls="--", label="DeepONet")
    ax.set_title(f"t = {t_fixed}"); ax.legend()
plt.tight_layout(); plt.show()

# ── BONUS: try a Gaussian bump IC ───────────────────
ic_bump  = np.exp(-2*sensor_pts**2)[np.newaxis, :]  # no retraining!
u_bump   = model.predict((ic_bump, xt)).flatten()
plt.plot(x_vals, u_bump, label="Gaussian bump IC, t=3")
plt.legend(); plt.show()